# BERT Approach

This notebook presents the bert based approach in solving the three way classification problem of clarity labeling

In [18]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch
from torch.utils.data import DataLoader, TensorDataset
import pandas as pd
from torch.optim import AdamW

In [23]:
df = pd.read_csv('../dataset/training_data_processed.csv')
drop_cols = ['evasion_label', 'Unnamed: 0', 'affirmative_questions']
df = df.drop(columns= drop_cols)
df.head()

,question_order,interview_question,interview_answer,gpt3.5_summary,gpt3.5_prediction,question,clarity_label
0,1,Q. Of the Biden administration. And accused th...,"Well, look, first of all, theI am sincere abou...",The question consists of 2 parts: \n1. How wou...,Question part: 1. How would you respond to the...,How would you respond to the accusation that t...,Clear Reply
1,1,Q. Of the Biden administration. And accused th...,"Well, look, first of all, theI am sincere abou...",The question consists of 2 parts: \n1. How wou...,Question part: 1. How would you respond to the...,Do you think President Xi is being sincere abo...,Ambivalent
2,2,Q. No worries. Do you believe the country's sl...,"Look, I think China has a difficult economic p...",The question consists of two parts:\n\n1. Q1: ...,Question part: Q1 - Do you believe the country...,Do you believe the country's slowdown and gro...,Ambivalent
3,2,Q. No worries. Do you believe the country's sl...,"Look, I think China has a difficult economic p...",The question consists of two parts:\n\n1. Q1: ...,Question part: Q1 - Do you believe the country...,Are you worried about the meeting between Pre...,Ambivalent
4,3,"Q. I can imagine. It is evening, I'd like to r...","Well, I hope I get to see Mr. Xi sooner than l...",The question consists of 3 parts:\n1. Is the P...,Question part: 1. Is the President's engagemen...,Is the President's engagement with Asian coun...,Clear Reply


In [32]:
label_map = {
    'Clear Reply': 0,
    'Clear Non-Reply': 1,
    'Ambivalent': 2
}
num_labels = len(label_map)
model_name = 'bert-base-uncased'


model = BertForSequenceClassification.from_pretrained(model_name, num_labels=num_labels) # For binary classification
tokenizer = BertTokenizer.from_pretrained(model_name)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [33]:
def data_prep(questions, answers, clarity_label):
    encodings = tokenizer(questions.tolist(), 
                          answers.tolist(), 
                          padding= True, 
                          truncation = True,
                          max_length = 512,
                          return_tensors = 'pt'
                          ) 
    label_tensors = torch.tensor([label_map[l] for l in clarity_label.tolist()])
    dataset = TensorDataset(
        encodings['input_ids'],
        encodings['attention_mask'],
        encodings['token_type_ids'],
        label_tensors
        )
    return dataset
    

In [34]:
BATCH_SIZE = 8

train_dataset = data_prep(df['question'], df['interview_answer'], df['clarity_label'])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.
Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pai

### Model Training

With preprocessing done we can now gop on to build the training loop to finetune the BERT model

In [35]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
epochs = 5
optimizer = AdamW(model.parameters(), lr = 5e-5)


model.train()

for epoch in range(epochs):
    print(f'Starting Epoch {epoch+1}')
    for batch in train_loader:
        input_ids = batch[0].to(device)
        attention_mask = batch[1].to(device)
        token_type_ids = batch[2].to(device)
        labels = batch[3].to(device)

        model.zero_grad()

        outputs = model(input_ids= input_ids,
                        attention_mask = attention_mask,
                        token_type_ids = token_type_ids,
                        labels = labels
                        )
        
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        optimizer.step()

        print(f'Epoch {epoch + 1} finished. Loss: {loss.item():.3f}')

Starting Epoch 1
Epoch 1 finished. Loss: 1.359


KeyboardInterrupt: 